## Import

In [8]:
import os
import gc
import math
import pickle

import warnings
from sklearn.model_selection import KFold

import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
import numpy as np
import pandas as pd
import torchvision.transforms as T
from box import Box
from timm import create_model
from torchvision.io import read_image
from torch.utils.data import DataLoader, Dataset 

import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, seed_everything
from pytorch_lightning.callbacks import ModelCheckpoint, TQDMProgressBar, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger

print(pl.__version__)
warnings.filterwarnings("ignore")

## Config

In [9]:
config = {'exp_name':'baseline_v1',
          'seed': 2023,
          'root': '../input/petfinder-pawpularity-score/', 
          'n_splits': 5,
          'n_epochs': 200,
          'early_stop': 30,
          'lr': 1e-3,
          'pretrain': True,
          'image_size': 384,
          'train_loader': {
              'batch_size': 16,
              'shuffle': True,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'val_loader': {
              'batch_size': 16,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'test_loader': {
              'batch_size': 32,
              'shuffle': False,
              'num_workers': os.cpu_count(),
              'pin_memory': True,
              'drop_last': False
          },
          'model':{
              'name': 'swin_large_patch4_window12_384',
              'output_dim': 1
          },
          'loss': 'nn.BCEWithLogitsLoss',
}

config = Box(config)

## Fix Seed

In [10]:
seed_everything(config.seed)

Global seed set to 2023


2023

## Tools

In [11]:
def mixup(x: torch.Tensor, y: torch.Tensor, alpha: float = 1.0):
    assert alpha > 0, "alpha should be larger than 0"
    assert x.size(0) > 1, "Mixup cannot be applied to a single instance."

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(x.size()[0])
    mixed_x = lam * x + (1 - lam) * x[rand_index, :]
    target_a, target_b = y, y[rand_index]
    return mixed_x, target_a, target_b, lam


def rmse(predict,target):
    return 100. * torch.sqrt(nn.MSELoss()(predict, target))

## Dataset

In [12]:
class PetfinderDataset(Dataset):
    """Dataset
    Args:
        df: the dataframe from csv, and the "Id" column needs to be the path of Image
    """
    def __init__(self, df, transform=None):
        # 判斷有沒有分數
        if "Pawpularity" in df.keys():
            # 取出分數
            self._y = df["Pawpularity"]
            
        self._X = df.drop("Pawpularity", axis=1)
            
    def __len__(self):
        return len(self._X)

    def __getitem__(self, idx):
        input_feature = self._X.iloc[idx].to_numpy().astype(np.float32)
        
        if self._y is not None:
            label = self._y.iloc[idx].astype(np.float32)
            return input_feature, label
        return input_feature
    
    
class PetfinderDataModule(LightningDataModule):
    """PetfinderDataModule
    Args:
        cfg (dict): config
        fold (int): idx of fold
        stage (srt): train or test
    """
    def __init__(self, cfg, fold, stage="train"):
        super().__init__()
        df = pd.read_csv("predicts.csv")
        df.drop("Id", axis=1, inplace=True)
        
        # self._df = df[:100]  # 用100筆資料測試訓練過程
        self._df = df
        self._cfg = cfg
        self.fold = fold

    def setup(self, stage: str):
        if (not os.path.isfile('temp.pickle')) or (self.fold == 0):
            # 創建kfold的dataframe index
            kf = KFold(n_splits=config.n_splits, shuffle=True, random_state=config.seed)
            all_splits = [(x.tolist(), y.tolist()) for x, y in kf.split(self._df)]

            # 將list of indexs存成緩存檔案
            with open('temp.pickle', 'wb') as f:
                pickle.dump(all_splits, f)
                
        # 讀取list of indexs，並取出本次fold資料分配
        with open('temp.pickle', 'rb') as f:
            all_splits = pickle.load(f)

        train_indexes, val_indexes = all_splits[self.fold]
        self.train_df, self.valid_df = self._df.iloc[train_indexes], self._df.iloc[val_indexes]

        # df to dataset
        self.train_data = PetfinderDataset(self.train_df)
        self.val_data = PetfinderDataset(self.valid_df)
                
    def train_dataloader(self):
        return DataLoader(self.train_data, **self._cfg.train_loader)
    
    def val_dataloader(self):
        return DataLoader(self.val_data, **self._cfg.val_loader)

In [13]:
fold = 0
a = PetfinderDataModule(config,fold)
a.setup('train')
a.train_data.__getitem__(0)[1].dtype

dtype('float32')

## Model

In [14]:
class Model(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.save_hyperparameters(hparams)  # 儲存超參數
        
        self._criterion = eval(self.hparams.loss)()
        self.metrics = rmse
        
        self.validation_step_outputs = {'val/logits': [],
                                        'val/pred': [],
                                        'val/labels': []}  # 用來計算epoch的val/loss, val/rmse
        
        self.__build_model()
        
    def __build_model(self):       
        # self.layer1 = nn.Linear(13, 128)
        # self.layer2 = nn.Linear(128, 64)
        # self.layer3 = nn.Linear(64, 1)
        self.fc = nn.Sequential(nn.Linear(13, 128),
                                nn.ReLU(),
                                nn.Linear(128, 64),
                                nn.ReLU(),
                                nn.Linear(64, 1))

    def forward(self, x):
        # out = self.layer1(x)
        # out = self.layer2(out)
        out = self.fc(x)
        return out

    def training_step(self, batch, batch_idx):
        images, labels = batch
    
        logits = self(images).squeeze(1)
        
        loss = self._criterion(logits, labels)
            
        self.log("train/loss", loss, prog_bar=True)
        return loss
        
    def validation_step(self, batch, batch_idx):
        images, labels = batch

        logits = self(images).squeeze(1)

        loss = self._criterion(logits, labels)
        
        self.validation_step_outputs['val/logits'].append(logits)
        self.validation_step_outputs['val/labels'].append(labels)
        
        return {'val/loss': loss}
    
    def on_validation_epoch_end(self):
        logits = torch.cat(self.validation_step_outputs['val/logits'], dim=0)
        labels = torch.cat(self.validation_step_outputs['val/labels'], dim=0)
        pred = torch.sigmoid(logits)
        
        loss = self._criterion(logits, labels)
        metric = self.metrics(pred, labels)
        
        self.log('val/loss', loss, prog_bar=True)
        self.log('val/metric', metric, prog_bar=True)
        
        self.validation_step_outputs['val/logits'].clear()
        self.validation_step_outputs['val/pred'].clear()
        self.validation_step_outputs['val/labels'].clear()

    def predict_step(self, batch, batch_idx):
        images, labels = batch
        pred = self(images).sigmoid().detach().cpu().numpy()
        return pred
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        lr_scheduler = ReduceLROnPlateau(optimizer,
                                         mode='min',
                                         factor=0.5,
                                         patience=10,
                                         min_lr=1e-7)
        lr_scheduler_config = {
            "scheduler": lr_scheduler,
            "interval": "epoch",
            "frequency": 1,
            "monitor": "train/loss",
        }
        return {
        "optimizer": optimizer,
        "lr_scheduler": lr_scheduler_config,
    }

## Train

### Callbacks

In [15]:
# 解決inf/nan無法於tqdm中顯示的問題，將其轉換為None
def convert_inf(x):
    if x is None or math.isinf(x) or math.isnan(x):
        return None
    return x


# 使TQDMProgressBar顯示fold的進度
class CustomTQDMProgressBar(TQDMProgressBar):
    def __init__(self, fold):
        super().__init__()
        self.fold = fold

    def on_train_epoch_start(self, trainer, pl_module):
        super().on_train_epoch_start(trainer, pl_module)
        self.train_progress_bar.reset(convert_inf(self.total_train_batches))
        self.train_progress_bar.initial = 0
        self.train_progress_bar.set_description(f"Fold {self.fold}, Epoch {trainer.current_epoch}")
        

# 將常用的參數設為預設（只是為了簡化kfold loop中的code）
class CustomModelCheckpoint(ModelCheckpoint):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            save_top_k=5,
            monitor="val/metric",
            mode="min",
            filename="metric={val/metric:.3f}-epoch={epoch:02d}-loss={val/loss:.3e}",
            auto_insert_metric_name=False,
            save_on_train_epoch_end=False,
            **kwargs
        )
        
        
class CustomEarlyStopping(EarlyStopping):
    def __init__(self, *args, **kwargs):
        super().__init__(
            *args,
            monitor="val/metric",
            patience=config.early_stop,
            mode="min",
            # verbose=False,
            **kwargs
        )

### Train Loop

In [9]:
warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

for fold in range(config.n_splits):  
    datamodule = PetfinderDataModule(config, fold=fold, stage="train")  # 載入data
    
    model = Model(config)  # 載入model

    # 宣告功能
    bar = CustomTQDMProgressBar(fold)
    tb_logger = TensorBoardLogger(save_dir='lightning_logs', name=f'fold_{fold}')
    checkpoint_callback = CustomModelCheckpoint(dirpath=tb_logger.log_dir)
    early_stop_callback = CustomEarlyStopping()
    lr_monitor = LearningRateMonitor(logging_interval='epoch')
    
    trainer = pl.Trainer(max_epochs=config.n_epochs, 
                        precision='16-mixed', # AMP 自動混合精度
                        benchmark=True,  # 加速運算
                        callbacks=[bar, checkpoint_callback, early_stop_callback, lr_monitor],  # 設定callback，可以指定多個callbacks
                        logger=tb_logger,
                        enable_model_summary=False, # 不顯示模型架構的資訊
                        # num_sanity_val_steps=1,  # 在正式訓練前先跑一次val_loop做測試，確保val_loop是正確的
                        )

    trainer.fit(model, datamodule=datamodule)
    
    # 刪除不必要的變數與cache，避免爆內存
    del datamodule, model, bar, tb_logger, checkpoint_callback, \
        early_stop_callback, trainer

    torch.cuda.empty_cache()
    gc.collect()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

## Test

In [19]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]  # RGB
IMAGENET_STD = [0.229, 0.224, 0.225]  # RGB

val_transform = T.Compose([T.Resize(config.image_size),
                            T.CenterCrop([config.image_size, config.image_size]),
                            T.ConvertImageDtype(torch.float),
                            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

df = pd.read_csv("predicts.csv")
df.drop("Id", axis=1, inplace=True)
# df to dataset
predict_data = PetfinderDataset(df)
predict_loader = DataLoader(predict_data, **config.test_loader)

### kFold predictions

In [20]:
warnings.filterwarnings("ignore")  # 關閉 Warning
torch.set_float32_matmul_precision('high')  # 設置高精度(根據顯卡調整)

total_predictions = []
for fold in range(config.n_splits):
    model = Model(config).load_from_checkpoint(f'lightning_logs/fold_{fold}/version_0/best.ckpt')

    trainer = pl.Trainer(logger=False)
    predictions = trainer.predict(model, dataloaders=predict_loader)
    
    total_predictions.append(np.concatenate(predictions).flatten())
    
    # 清除變數與快取
    del model, trainer, predictions
    torch.cuda.empty_cache()
    gc.collect()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: 0it [00:00, ?it/s]

### 計算平均預測分數

In [21]:
mean_predictions = np.array(total_predictions).mean(axis=0)
print(mean_predictions)

[0.4122035  0.4585201  0.4551723  ... 0.25317854 0.38140935 0.545295  ]


### 儲存平均預測分數

In [ ]:
# 儲存csv
mean_predicts_df = pd.DataFrame(mean_predictions, columns=["mean_predicts_fine"])
df = pd.concat([df, mean_predicts_df], axis=1)
df.to_csv("predicts_fine.csv", index=False)

# 存成pickle
with open('predicts_fine.pickle', 'wb') as f:
    pickle.dump(mean_predictions, f)

### 計算RMSE

In [23]:
def rmse(predict,target):
    return (((predict - target) ** 2).mean() ** 0.5)

metric = 100. * rmse(mean_predictions, df["Pawpularity"].to_numpy())
print(f"metric: {metric}")

metric: 15.475099500443553
